# RLI Assignment 22.00: Mountain Car

## Setup & Dependencies


In [ ]:
# Uncomment to install dependencies (all should be present in the project venv)
# !pip install gymnasium stable-baselines3 torch matplotlib numpy pandas seaborn tensorboard

# Library versions used:
# gymnasium==1.2.3, stable-baselines3==2.8.0, torch==2.10.0, numpy==2.2.6


In [1]:
import sys
sys.path.insert(0, '.')

import mclib as mc

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
import seaborn as sns
import time, random, warnings
warnings.filterwarnings('ignore')

import gymnasium as gym
import torch

from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 4)

print("Setup complete.")


Setup complete.


## 1. Environment Analysis

### 1.1 The Mountain Car Problem

In [ ]:
# Explore both environment versions
env_disc = gym.make('MountainCar-v0')
env_cont = gym.make('MountainCarContinuous-v0')

print("DISCRETE: MountainCar-v0")
print(f"Action space: {env_disc.action_space}")
print(f"Observation space: {env_disc.observation_space}")
print(f"Obs low: {env_disc.observation_space.low}")
print(f"Obs high: {env_disc.observation_space.high}")

print("\nCONTINUOUS: MountainCarContinuous-v0")
print(f"Action space: {env_cont.action_space}")
print(f"Observation space: {env_cont.observation_space}")

env_disc.close()
env_cont.close()


In [ ]:
# Visualize the landscape and phase portrait concept
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Mountain Car landscape
x = np.linspace(-1.2, 0.6, 400)
y = np.sin(3 * x)  # height profile

ax = axes[0]
ax.plot(x, y, 'k-', linewidth=2, label='Terrain: sin(3x)')
ax.axvline(-0.5, color='blue', linestyle='--', alpha=0.5, label='Start region [-0.6, -0.4]')
ax.axvline(0.5,  color='green', linestyle='--', alpha=0.7, label='Goal (discrete) x>=0.5')
ax.axvline(0.45, color='lime',  linestyle=':',  alpha=0.7, label='Goal (continuous) x>=0.45')
ax.scatter([-0.5], [np.sin(-1.5)], s=200, c='blue',  zorder=5, label='Car start (example)')
ax.scatter([0.5],  [np.sin(1.5)],  s=200, c='gold', marker='^', zorder=5, label='Flag (discrete)')
ax.set_xlabel('Position x')
ax.set_ylabel('Height sin(3x)')
ax.set_title('Mountain Car: Landscape')
ax.legend(fontsize=8, loc='upper left')

# Right: State space boundaries
ax2 = axes[1]
ax2.set_xlim(-1.25, 0.65)
ax2.set_ylim(-0.075, 0.075)

# Shade the starting zone
ax2.axvspan(-0.6, -0.4, alpha=0.2, color='blue', label='Start zone')
ax2.axvspan(0.45, 0.6, alpha=0.2, color='green', label='Goal zone (continuous)')
ax2.axvspan(0.5, 0.6, alpha=0.3, color='lime', label='Goal zone (discrete)')
ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax2.axvline(-1.2, color='red', linestyle=':', label='Left wall')
ax2.set_xlabel('Position x [-1.2, 0.6]')
ax2.set_ylabel('Velocity v [-0.07, 0.07]')
ax2.set_title('State Space (Position x Velocity)')
ax2.legend(fontsize=8)

plt.suptitle('Mountain Car Environment Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Scenario 1: Discrete Minimum Steps

**Design Choices**
- State representation: 
- Rewards:
- Algorithms:
- Training strategy: 
- Custom Wrappers: 
- Hyperparameters:
- Objective performance vs engineered reward:

In [ ]:
# Scenario 1 — sanity check: MountainCar-v0 default reward
env = mc.make_s1()
obs, _ = env.reset(seed=0)
print(f"Observation space : {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Initial obs: {obs}")
obs2, r, term, trunc, _ = env.step(2)
print(f"Default reward for action=2: {r}  (should be -1.0)")
env.close()


### Model 1: Tabular Q-Learning (default hyperparameters)

In [ ]:
s1_ql_agent = mc.TabularQLearningAgent(n_bins=40, alpha=0.15, gamma=0.99, eps_start=1.0,  eps_end=0.01, eps_decay=0.9997 )

with mc.Testbed('S1/QL_default', log_dir='runs') as tb:
    s1_ql_results = tb.run_tabular(mc.make_s1, s1_ql_agent, n_episodes=15_000, eval_every=2_000,n_eval=50, verbose=True)

s1_ql_rewards = s1_ql_results['rewards']
s1_ql_eval_means = s1_ql_results['eval_means']
s1_ql_eval_stds = s1_ql_results['eval_stds']
s1_ql_eval_eps = s1_ql_results['eval_episodes']
s1_ql_final = s1_ql_results['final']
print(f"\nQL default — mean={np.mean(s1_ql_final):.2f} +/- {np.std(s1_ql_final):.2f} | "
      f"success={np.mean(np.array(s1_ql_final) > -200):.1%}")


In [ ]:
# Launch TensorBoard to compare all S1 runs
# Run this in a terminal (not in the notebook):
#      tensorboard --logdir runs
# Then open http://localhost:6006 in your browser.


## 3. Scenario 2: Continuous and Minimum Fuel


**Design Choices**
- State representation: 
- Rewards:
- Algorithms:
- Training strategy: 
- Custom Wrappers: 
- Hyperparameters:
- Objective performance vs engineered reward:

## 4. Scenario 3: Discrete Adapted and Minimum Fuel

**Design Choices**
- State representation: 
- Rewards:
- Algorithms:
- Training strategy: 
- Custom Wrappers: 
- Hyperparameters:
- Objective performance vs engineered reward:

## 5. Scenario 4: Continuous Adapted Action Use


**Design Choices**
- State representation: continuous state `[position, velocity]` from `MountainCarContinuous-v0`; no feature engineering during training.
- Reward hierarchy: A is a minimum-time baseline, B is the standard continuous squared-action fuel baseline, C is the pure Scenario 4 non-null action-use objective, and D is a shaped action-use training reward.
- Main Scenario 4 objective: evaluate every trained policy under the same non-null action-use reward so the final comparison is based on success rate, mean steps, fuel `sum(action^2)`, linear effort `sum(abs(action))`, and non-null action count.
- Algorithms: SAC and TD3, both using `MlpPolicy` for continuous control.
- Training strategy: independent runs with the same seed; SAC runs use 150,000 timesteps and TD3 runs use 300,000 timesteps.
- TD3 setup: Ornstein-Uhlenbeck exploration noise with `sigma=0.5`, larger replay buffer, smaller batch size, and RL-Zoo-style update settings for this environment.
- Custom wrappers: `ContinuousStepsWrapper`, `ContinuousActionUseWrapper`, and `ContinuousShapedRewardWrapper`, exposed through `mc.make_s4()`, `mc.make_s4_action_use()`, and `mc.make_s4_shaped()`.
- Interpretation: C is the pure scenario reward. D is a training aid for the deceptive sparse-reward setting, so shaped policies are still judged with the pure Scenario 4 objective metrics.


In [2]:
# Scenario 4 reward sanity checks
s4_env = mc.make_s4()
s4_default_env = mc.make_s4_default()
s4_action_env = mc.make_s4_action_use()
s4_shaped_env = mc.make_s4_shaped()

print("S4 step env")
print(f"Observation space: {s4_env.observation_space}")
print(f"Action space: {s4_env.action_space}")
print(f"Max episode steps: {s4_env.spec.max_episode_steps}")

obs, _ = s4_env.reset(seed=SEED)
_, r_step, term_step, trunc_step, _ = s4_env.step(np.array([0.5], dtype=np.float32))
print(f"Step reward example action=0.5: {r_step:.3f} | terminated={term_step} truncated={trunc_step}")
s4_env.close()

obs, _ = s4_default_env.reset(seed=SEED)
_, r_default, term_default, trunc_default, _ = s4_default_env.step(np.array([0.5], dtype=np.float32))
expected_default = -0.1 * (0.5 ** 2)
print("\nS4 default continuous env")
print(f"Default reward example action=0.5: {r_default:.3f} | expected {expected_default:.3f}")
s4_default_env.close()

obs, _ = s4_action_env.reset(seed=SEED)
obs2, r_action, term_action, trunc_action, info_action = s4_action_env.step(np.array([0.5], dtype=np.float32))
expected_action = (
    -mc.ContinuousActionUseWrapper.TIME_PENALTY
    - mc.ContinuousActionUseWrapper.ACTION_USE_COST
    - mc.ContinuousActionUseWrapper.ACTION_MAG_COST * 0.5
)
print("\nS4 action-use env")
print(f"Action-use reward example action=0.5: {r_action:.3f} | expected {expected_action:.3f}")
print(f"Logged action abs: {info_action.get('s4_action_abs'):.3f}")
print(f"Logged non-null action: {info_action.get('s4_non_null_action')}")
s4_action_env.close()

obs, _ = s4_shaped_env.reset(seed=SEED)
obs2, r_shaped, term_shaped, trunc_shaped, info_shaped = s4_shaped_env.step(np.array([0.5], dtype=np.float32))
progress_bonus = mc.ContinuousShapedRewardWrapper.PROGRESS_WEIGHT * float(obs2[0] - obs[0])
velocity_bonus = mc.ContinuousShapedRewardWrapper.VELOCITY_WEIGHT * abs(float(obs2[1]))
expected_shaped = expected_action + progress_bonus + velocity_bonus
print("\nS4 shaped action-use env")
print(f"Shaped reward example action=0.5: {r_shaped:.3f} | expected {expected_shaped:.3f}")
print(f"Progress bonus: {info_shaped.get('s4_progress_bonus'):.6f}")
print(f"Velocity bonus: {info_shaped.get('s4_velocity_bonus'):.6f}")
s4_shaped_env.close()


S4 step env
Observation space: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
Action space: Box(-1.0, 1.0, (1,), float32)
Max episode steps: 999
Step reward example action=0.5: -1.000 | terminated=False truncated=False

S4 default continuous env
Default reward example action=0.5: -0.025 | expected -0.025

S4 action-use env
Action-use reward example action=0.5: -0.120 | expected -0.120
Logged action abs: 0.500
Logged non-null action: True

S4 shaped action-use env
Shaped reward example action=0.5: -0.120 | expected -0.120
Progress bonus: 0.000167
Velocity bonus: 0.000017


### Scenario 4 Experiments

The runs below implement the adjusted hierarchy from the literature review: minimum-time baseline, default continuous squared-action baseline, pure non-null action-use reward, and shaped non-null action-use training reward. All final summaries use the pure Scenario 4 action-use environment for evaluation, even when the training reward was the default or shaped variant. This keeps the report focused on the assignment objective while still showing how engineered rewards affect learning.


In [ ]:
S4_SAC_TOTAL_TIMESTEPS = mc.S4_TOTAL_TIMESTEPS
S4_TD3_TOTAL_TIMESTEPS = mc.S4_TD3_TOTAL_TIMESTEPS
S4_EVAL_EPISODES = 100
S4_EVAL_FREQ = 10_000
S4_N_EVAL_DURING_TRAINING = 20
S4_OBJECTIVE_EVAL_ENV_FACTORY = mc.make_s4_action_use

s4_reward_setups = [
    {
        "run_key": "step",
        "reward_design": "A: minimum-time baseline",
        "env_factory": mc.make_s4,
    },
    {
        "run_key": "default",
        "reward_design": "B: default continuous squared-action baseline",
        "env_factory": mc.make_s4_default,
    },
    {
        "run_key": "action_use",
        "reward_design": "C: pure non-null action-use objective",
        "env_factory": mc.make_s4_action_use,
    },
    {
        "run_key": "shaped",
        "reward_design": "D: shaped action-use training reward",
        "env_factory": mc.make_s4_shaped,
    },
]

s4_algorithm_setups = [
    {"algorithm": "SAC", "total_timesteps": S4_SAC_TOTAL_TIMESTEPS},
    {"algorithm": "TD3", "total_timesteps": S4_TD3_TOTAL_TIMESTEPS},
]

s4_run_configs = []
for reward_setup in s4_reward_setups:
    for algorithm_setup in s4_algorithm_setups:
        s4_run_configs.append({
            "run_name": f"S4_{reward_setup['run_key']}_{algorithm_setup['algorithm']}",
            "algorithm": algorithm_setup["algorithm"],
            "reward_design": reward_setup["reward_design"],
            "env_factory": reward_setup["env_factory"],
            "objective_env_factory": S4_OBJECTIVE_EVAL_ENV_FACTORY,
            "total_timesteps": algorithm_setup["total_timesteps"],
        })

def s4_agent_fn(model):
    def _act(obs):
        action, _ = model.predict(obs, deterministic=True)
        return action
    return _act

print(f"Configured {len(s4_run_configs)} Scenario 4 runs.")
print(f"Reward setups: {len(s4_reward_setups)} x algorithms: {len(s4_algorithm_setups)}")
print(f"SAC timesteps: {S4_SAC_TOTAL_TIMESTEPS:,}")
print("SAC settings: RL-Zoo style fixed ent_coef=0.1, gamma=0.9999, use_sde=True")
print(f"TD3 timesteps: {S4_TD3_TOTAL_TIMESTEPS:,}")
print(f"TD3 gamma: {mc.TD3_GAMMA}")
print(f"TD3 learning starts: {mc.TD3_LEARNING_STARTS:,}")
print(f"TD3 action noise: {mc.TD3_ACTION_NOISE_TYPE}, sigma={mc.TD3_ACTION_NOISE_SIGMA}")
print(f"S4 action epsilon: {mc.ContinuousActionUseWrapper.ACTION_EPSILON}")
print(f"S4 time penalty: {mc.ContinuousActionUseWrapper.TIME_PENALTY}")
print(f"S4 action-use cost: {mc.ContinuousActionUseWrapper.ACTION_USE_COST}")
print(f"S4 action-magnitude cost: {mc.ContinuousActionUseWrapper.ACTION_MAG_COST}")
print(f"S4 shaped progress weight: {mc.ContinuousShapedRewardWrapper.PROGRESS_WEIGHT}")
print(f"S4 shaped velocity weight: {mc.ContinuousShapedRewardWrapper.VELOCITY_WEIGHT}")


In [ ]:
# This cell performs the Scenario 4 training/evaluation sweep.
# Existing in-memory or saved models are reused. Missing models are trained once and saved.
from pathlib import Path

S4_MODEL_DIR = Path("models/scenario4")
S4_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if "s4_results" not in globals():
    s4_results = {}
else:
    s4_results = dict(s4_results)


def _empty_eval_trace():
    return {
        "eval_timesteps": [],
        "eval_mean_rewards": [],
        "eval_std_rewards": [],
        "eval_success_rates": [],
        "eval_mean_steps": [],
    }


for cfg in s4_run_configs:
    run_name = cfg["run_name"]
    model_path = S4_MODEL_DIR / f"{run_name}.zip"
    result = s4_results.get(run_name)

    if result is not None and "model" in result:
        print(f"\n=== Reusing in-memory {run_name} ===")
        train_result = {
            "model": result["model"],
            "run_name": run_name,
            "log_dir": result.get("log_dir", str(Path("runs") / run_name)),
            **{key: result.get(key, []) for key in _empty_eval_trace()},
        }
        if not model_path.exists():
            train_result["model"].save(model_path)
            print(f"Saved reusable model to {model_path}")
    elif model_path.exists():
        print(f"\n=== Loading saved {run_name} ===")
        model = mc.load_sb3_continuous_model(
            cfg["algorithm"],
            model_path,
            cfg["env_factory"],
            seed=SEED,
            verbose=0,
        )
        train_result = {
            "model": model,
            "run_name": run_name,
            "log_dir": str(Path("runs") / run_name),
            **_empty_eval_trace(),
        }
    else:
        print(f"\n=== Training missing {run_name} ===")
        train_result = mc.train_sb3_continuous(
            cfg["algorithm"],
            cfg["env_factory"],
            run_name=run_name,
            total_timesteps=cfg["total_timesteps"],
            seed=SEED,
            tensorboard_log="runs",
            eval_freq=S4_EVAL_FREQ,
            n_eval_episodes=S4_N_EVAL_DURING_TRAINING,
            verbose=0,
            model_save_path=model_path,
        )
        print(f"Saved reusable model to {model_path}")

    objective_eval = mc.evaluate_continuous_policy(
        train_result["model"],
        cfg["objective_env_factory"],
        n_episodes=S4_EVAL_EPISODES,
        deterministic=True,
    )
    if cfg["env_factory"] is cfg["objective_env_factory"]:
        training_reward_eval = objective_eval
    else:
        training_reward_eval = mc.evaluate_continuous_policy(
            train_result["model"],
            cfg["env_factory"],
            n_episodes=S4_EVAL_EPISODES,
            deterministic=True,
        )
    s4_results[run_name] = {
        **cfg,
        **train_result,
        "model_path": str(model_path),
        "objective_eval": objective_eval,
        "training_reward_eval": training_reward_eval,
        "final_eval": objective_eval,
    }
    objective_summary = objective_eval["summary"]
    training_summary = training_reward_eval["summary"]
    print(
        f"{run_name} | objective reward={objective_summary['mean_reward']:.2f} +/- {objective_summary['std_reward']:.2f} | "
        f"training reward={training_summary['mean_reward']:.2f} | success={objective_summary['success_rate']:.1%} | "
        f"mean steps={objective_summary['mean_steps']:.1f} | non-null actions={objective_summary['mean_non_null_actions']:.1f}"
    )


In [ ]:
# Numerical and statistical summary over 100 deterministic objective-evaluation episodes.
s4_summary_rows = []
for cfg in s4_run_configs:
    run_name = cfg["run_name"]
    objective_summary = s4_results[run_name]["objective_eval"]["summary"]
    training_summary = s4_results[run_name]["training_reward_eval"]["summary"]
    s4_summary_rows.append({
        "run": run_name,
        "algorithm": cfg["algorithm"],
        "reward_design": cfg["reward_design"],
        "objective_mean_reward": objective_summary["mean_reward"],
        "objective_std_reward": objective_summary["std_reward"],
        "objective_reward_ci95": objective_summary["reward_ci95"],
        "training_eval_mean_reward": training_summary["mean_reward"],
        "success_rate": objective_summary["success_rate"],
        "mean_steps": objective_summary["mean_steps"],
        "steps_ci95": objective_summary["steps_ci95"],
        "mean_steps_success": objective_summary["mean_steps_success"],
        "mean_fuel": objective_summary["mean_fuel"],
        "mean_linear_effort": objective_summary["mean_linear_effort"],
        "mean_non_null_actions": objective_summary["mean_non_null_actions"],
        "mean_abs_action": objective_summary["mean_abs_action"],
        "mean_max_abs_action": objective_summary["mean_max_abs_action"],
    })

s4_summary = pd.DataFrame(s4_summary_rows).sort_values(
    ["success_rate", "mean_steps", "mean_non_null_actions"], ascending=[False, True, True]
).reset_index(drop=True)
s4_summary


In [ ]:
# TensorBoard: run `tensorboard --logdir runs` for full SB3 diagnostics.
# These inline curves show the periodic deterministic evaluation trace collected during training.
n_cols = 2
n_rows = int(np.ceil(len(s4_run_configs) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.5 * n_rows), sharex=False)
axes = np.asarray(axes).reshape(-1)
for ax, cfg in zip(axes, s4_run_configs):
    data = s4_results[cfg["run_name"]]
    ax.errorbar(
        data["eval_timesteps"],
        data["eval_mean_rewards"],
        yerr=data["eval_std_rewards"],
        fmt="o-",
        capsize=3,
    )
    ax.set_title(cfg["run_name"])
    ax.set_xlabel("Training timesteps")
    ax.set_ylabel("Training-reward eval")
    ax.grid(True, alpha=0.3)
for ax in axes[len(s4_run_configs):]:
    ax.axis("off")
plt.suptitle("Scenario 4: Evaluation Curves During Training", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Continuous policy maps: signed force and force magnitude across the state space.
s4_action_grids = {}
fig, axes = plt.subplots(len(s4_run_configs), 2, figsize=(13, 3.4 * len(s4_run_configs)))
axes = np.asarray(axes)
for row, cfg in enumerate(s4_run_configs):
    run_name = cfg["run_name"]
    grid = mc.get_continuous_policy_grid(s4_results[run_name]["model"], n_bins=50)
    s4_action_grids[run_name] = grid
    mc.plot_continuous_action_heatmap(
        grid, n_bins=50, title=f"{run_name}: signed action", ax=axes[row, 0], magnitude=False
    )
    mc.plot_continuous_action_heatmap(
        grid, n_bins=50, title=f"{run_name}: action magnitude", ax=axes[row, 1], magnitude=True
    )
plt.tight_layout()
plt.show()


In [ ]:
# Phase portraits show whether each policy learns the left-right oscillation needed for momentum.
n_cols = 2
n_rows = int(np.ceil(len(s4_run_configs) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4.5 * n_rows), sharex=True, sharey=True)
axes = np.asarray(axes).reshape(-1)
for ax, cfg in zip(axes, s4_run_configs):
    run_name = cfg["run_name"]
    trajectories, rewards = mc.collect_trajectories(
        S4_OBJECTIVE_EVAL_ENV_FACTORY,
        s4_agent_fn(s4_results[run_name]["model"]),
        n_episodes=5,
        max_steps=mc.S4_MAX_STEPS,
    )
    mc.plot_phase_portrait(
        trajectories, rewards, title=run_name, ax=ax, goal_position=0.45
    )
for ax in axes[len(s4_run_configs):]:
    ax.axis("off")
plt.suptitle("Scenario 4: Phase Portraits Under Objective Reward", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Distribution checks: high variability means an agent is unreliable even when the mean looks good.
s4_episode_rows = []
for cfg in s4_run_configs:
    run_name = cfg["run_name"]
    for ep in s4_results[run_name]["objective_eval"]["episodes"]:
        s4_episode_rows.append({**ep, "run": run_name, "algorithm": cfg["algorithm"], "reward_design": cfg["reward_design"]})

s4_episode_df = pd.DataFrame(s4_episode_rows)

run_order = [cfg["run_name"] for cfg in s4_run_configs]

def s4_short_label(run_name):
    reward_name, algorithm = run_name.replace("S4_", "").rsplit("_", 1)
    reward_name = reward_name.replace("action_use", "action-use")
    return f"{reward_name}\n{algorithm}"

label_order = [s4_short_label(run_name) for run_name in run_order]
label_map = dict(zip(run_order, label_order))
s4_episode_df["run_label"] = pd.Categorical(
    s4_episode_df["run"].map(label_map), categories=label_order, ordered=True
)

fig, axes = plt.subplots(2, 2, figsize=(16, 9.5))
plot_specs = [
    {
        "metric": "steps",
        "title": "Steps to Goal / Truncation",
        "ylabel": "steps (zoomed)",
        "ylim": (55, 140),
        "note": "Counts above the axis are clipped failed/slow episodes.",
    },
    {
        "metric": "non_null_actions",
        "title": "Non-null Engine Actions",
        "ylabel": "non-null actions (zoomed)",
        "ylim": (0, 140),
        "note": "Counts above the axis are clipped high-use episodes.",
    },
    {
        "metric": "linear_effort",
        "title": "Linear Effort: sum(abs(action))",
        "ylabel": "linear effort",
        "ylim": (0, 95),
        "note": None,
    },
    {
        "metric": "fuel",
        "title": "Fuel Used: sum(action^2)",
        "ylabel": "fuel",
        "ylim": (0, 80),
        "note": None,
    },
]

for ax, spec in zip(axes.flat, plot_specs):
    metric = spec["metric"]
    y_low, y_high = spec["ylim"]
    sns.boxplot(
        data=s4_episode_df,
        x="run_label",
        y=metric,
        order=label_order,
        ax=ax,
        showfliers=True,
    )
    ax.set_ylim(y_low, y_high)
    ax.set_title(spec["title"])
    ax.set_xlabel("")
    ax.set_ylabel(spec["ylabel"])
    ax.tick_params(axis="x", rotation=0, labelsize=9)
    ax.grid(True, axis="y", alpha=0.3)

    clipped_counts = (
        s4_episode_df.groupby("run_label", observed=False)[metric]
        .apply(lambda values: int((values > y_high).sum()))
        .reindex(label_order, fill_value=0)
    )
    for x_pos, clipped in enumerate(clipped_counts):
        if clipped:
            ax.scatter(x_pos, y_high, marker="v", color="crimson", s=35, clip_on=False, zorder=5)
            ax.text(
                x_pos,
                y_high,
                f"{clipped} > {y_high:g}",
                color="crimson",
                ha="center",
                va="bottom",
                fontsize=8,
                rotation=90,
                clip_on=False,
            )

    if spec["note"]:
        ax.text(
            0.01,
            0.98,
            spec["note"],
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            color="dimgray",
        )

plt.suptitle("Scenario 4 Objective-Evaluation Distributions", fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
# Explanation bridge: discretize continuous actions into left / idle / right and fit the same regression explainer.
def discretize_continuous_action_grid(action_grid, threshold=0.05):
    return np.where(action_grid < -threshold, 0, np.where(action_grid > threshold, 2, 1))

s4_explanations = {}
for run_name, grid in s4_action_grids.items():
    discrete_policy = discretize_continuous_action_grid(grid, threshold=0.05)
    acc, weights, feature_names = mc.explain_policy_regression(
        discrete_policy, n_bins=grid.shape[0], scenario_name=run_name
    )
    s4_explanations[run_name] = {
        "accuracy": acc,
        "weights": weights,
        "feature_names": feature_names,
    }

## 6. Comparative Analysis

## 7. Explanation Tools